*HR Improvement*

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

# =========================
# 1. Load files
# =========================
file1 = "HR1.csv"   # Rater 1
file2 = "HR2.csv"   # Rater 2

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# =========================
# 2. Clean columns and rows
# =========================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

df1 = df1[df1["ID"].notna()].copy()
df2 = df2[df2["ID"].notna()].copy()

df1["ID"] = df1["ID"].astype(str).str.strip()
df2["ID"] = df2["ID"].astype(str).str.strip()

# =========================
# 3. Match common IDs
# =========================
df1 = df1.set_index("ID")
df2 = df2.set_index("ID")

common_ids = sorted(df1.index.intersection(df2.index))

df1 = df1.loc[common_ids].copy()
df2 = df2.loc[common_ids].copy()

print("Matched IDs:", len(common_ids))
print("Common IDs:", common_ids)

# =========================
# 4. Define dimensions
# =========================
dimensions = ["HRO", "MA", "PPA", "SPQ", "CER"]

# =========================
# TASK 1: Rater 1 averages
# =========================
rows_task1 = []

for dim in dimensions:
    our_avg = df1[f"{dim}_Our"].mean()
    base_avg = df1[f"{dim}_Base"].mean()

    rows_task1.append({
        "Dimension": dim,
        "Rater1_Our_Avg": our_avg,
        "Rater1_Base_Avg": base_avg
    })

task1_df = pd.DataFrame(rows_task1).round(3)

print("\n=== TASK 1: Rater 1 average for each dimension ===")
display(task1_df)

# =========================
# TASK 2: Rater 2 averages
# =========================
rows_task2 = []

for dim in dimensions:
    our_avg = df2[f"{dim}_Our"].mean()
    base_avg = df2[f"{dim}_Base"].mean()

    rows_task2.append({
        "Dimension": dim,
        "Rater2_Our_Avg": our_avg,
        "Rater2_Base_Avg": base_avg
    })

task2_df = pd.DataFrame(rows_task2).round(3)

print("\n=== TASK 2: Rater 2 average for each dimension ===")
display(task2_df)

# =========================
# TASK 3 + TASK 4 together
# Per-dimension comparison across both raters
# and improvement %
# =========================
rows_summary = []

for dim in dimensions:
    r1_our = df1[f"{dim}_Our"].mean()
    r2_our = df2[f"{dim}_Our"].mean()
    our_avg = (r1_our + r2_our) / 2

    r1_base = df1[f"{dim}_Base"].mean()
    r2_base = df2[f"{dim}_Base"].mean()
    base_avg = (r1_base + r2_base) / 2

    improvement_pct = ((our_avg - base_avg) / base_avg) * 100

    rows_summary.append({
        "Dimension": dim,
        "Rater1_Our_Avg": r1_our,
        "Rater2_Our_Avg": r2_our,
        "System_Avg": our_avg,
        "Rater1_Base_Avg": r1_base,
        "Rater2_Base_Avg": r2_base,
        "Baseline_Avg": base_avg,
        "Improvement_%": improvement_pct
    })

summary_df = pd.DataFrame(rows_summary)
summary_df = summary_df.round({
    "Rater1_Our_Avg": 3,
    "Rater2_Our_Avg": 3,
    "System_Avg": 3,
    "Rater1_Base_Avg": 3,
    "Rater2_Base_Avg": 3,
    "Baseline_Avg": 3,
    "Improvement_%": 1
})

print("\n=== TASK 3 + TASK 4: Per-dimension averages and improvement ===")
display(summary_df)

# =========================
# Optional: add Raw Total row
# =========================
r1_total_our = df1["Total_Our"].mean()
r2_total_our = df2["Total_Our"].mean()
system_total_avg = (r1_total_our + r2_total_our) / 2

r1_total_base = df1["Total_Base"].mean()
r2_total_base = df2["Total_Base"].mean()
baseline_total_avg = (r1_total_base + r2_total_base) / 2

total_improvement = ((system_total_avg - baseline_total_avg) / baseline_total_avg) * 100

total_row = pd.DataFrame([{
    "Dimension": "Raw Total",
    "Rater1_Our_Avg": round(r1_total_our, 3),
    "Rater2_Our_Avg": round(r2_total_our, 3),
    "System_Avg": round(system_total_avg, 3),
    "Rater1_Base_Avg": round(r1_total_base, 3),
    "Rater2_Base_Avg": round(r2_total_base, 3),
    "Baseline_Avg": round(baseline_total_avg, 3),
    "Improvement_%": round(total_improvement, 1)
}])

final_summary_df = pd.concat([summary_df, total_row], ignore_index=True)

print("\n=== Final Summary with Raw Total ===")
display(final_summary_df)

# =========================
# Save outputs
# =========================
with pd.ExcelWriter("HR_5dimension_avg_improvement.xlsx", engine="openpyxl") as writer:
    task1_df.to_excel(writer, sheet_name="Task1_Rater1_Avg", index=False)
    task2_df.to_excel(writer, sheet_name="Task2_Rater2_Avg", index=False)
    summary_df.to_excel(writer, sheet_name="Task3_4_ByDimension", index=False)
    final_summary_df.to_excel(writer, sheet_name="Final_With_Total", index=False)

print("\nSaved: HR_5dimension_avg_improvement.xlsx")

Matched IDs: 10
Common IDs: ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']

=== TASK 1: Rater 1 average for each dimension ===


,Dimension,Rater1_Our_Avg,Rater1_Base_Avg
0,HRO,20.400,14.590
1,MA,7.900,5.700
2,PPA,13.900,12.200
3,SPQ,8.400,6.200
4,CER,8.267,5.204



=== TASK 2: Rater 2 average for each dimension ===


,Dimension,Rater2_Our_Avg,Rater2_Base_Avg
0,HRO,20.820,14.730
1,MA,8.800,6.400
2,PPA,12.700,10.600
3,SPQ,8.400,6.100
4,CER,8.335,4.671



=== TASK 3 + TASK 4: Per-dimension averages and improvement ===


,Dimension,Rater1_Our_Avg,Rater2_Our_Avg,System_Avg,Rater1_Base_Avg,Rater2_Base_Avg,Baseline_Avg,Improvement_%
0,HRO,20.400,20.820,20.610,14.590,14.730,14.660,40.6
1,MA,7.900,8.800,8.350,5.700,6.400,6.050,38.0
2,PPA,13.900,12.700,13.300,12.200,10.600,11.400,16.7
3,SPQ,8.400,8.400,8.400,6.200,6.100,6.150,36.6
4,CER,8.267,8.335,8.301,5.204,4.671,4.938,68.1



=== Final Summary with Raw Total ===


,Dimension,Rater1_Our_Avg,Rater2_Our_Avg,System_Avg,Rater1_Base_Avg,Rater2_Base_Avg,Baseline_Avg,Improvement_%
0,HRO,20.400,20.820,20.610,14.590,14.730,14.660,40.6
1,MA,7.900,8.800,8.350,5.700,6.400,6.050,38.0
2,PPA,13.900,12.700,13.300,12.200,10.600,11.400,16.7
3,SPQ,8.400,8.400,8.400,6.200,6.100,6.150,36.6
4,CER,8.267,8.335,8.301,5.204,4.671,4.938,68.1
5,Raw Total,58.867,59.055,58.961,43.894,42.501,43.198,36.5



Saved: HR_5dimension_avg_improvement.xlsx


In [3]:
!pip install pingouin -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 5.4 MB/s eta 0:00:00


# **HR! vs HR2 interRater**

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import pingouin as pg

# =========================
# 2. Load files
# =========================
file1 = "HR1.csv"   # HR1
file2 = "HR2.csv"   # HR2

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# =========================
# 3. Clean data
# =========================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

df1 = df1[df1["ID"].notna()].copy()
df2 = df2[df2["ID"].notna()].copy()

df1["ID"] = df1["ID"].astype(str).str.strip()
df2["ID"] = df2["ID"].astype(str).str.strip()

# =========================
# 4. Match common IDs
# =========================
df1 = df1.set_index("ID")
df2 = df2.set_index("ID")

common_ids = sorted(df1.index.intersection(df2.index))

df1 = df1.loc[common_ids].copy()
df2 = df2.loc[common_ids].copy()

print("Matched IDs:", len(common_ids))
print("Common IDs:", common_ids)

# =========================
# 5. Define dimensions
# =========================
dims = ["HRO", "MA", "PPA", "SPQ", "CER"]

# =========================
# Helper function
# =========================
def compute_irr(r1_scores, r2_scores, unit_ids, measure_name):
    # Pearson
    pearson_r, pearson_p = pearsonr(r1_scores, r2_scores)

    # Spearman
    spearman_r, spearman_p = spearmanr(r1_scores, r2_scores)

    # ICC dataframe
    icc_df = pd.DataFrame({
        "ID": unit_ids * 2,
        "Rater": ["HR1"] * len(unit_ids) + ["HR2"] * len(unit_ids),
        "Score": list(r1_scores) + list(r2_scores)
    })

    icc_all = pg.intraclass_corr(
        data=icc_df,
        targets="ID",
        raters="Rater",
        ratings="Score"
    )
    print(f"\nColumns of icc_all for {measure_name}: {icc_all.columns.tolist()}\n")

    icc_all = icc_all[["Type", "ICC", "F", "df1", "df2", "pval", "CI95"]].copy()

    summary = pd.DataFrame([{
        "Measure": measure_name,
        "N": len(unit_ids),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p
    }])

    return summary, icc_all

# =========================================================
# PART 1: Total pooled combined (Our + Base)
# =========================================================
r1_total_combined = np.concatenate([
    df1["Total_Our"].astype(float).values,
    df1["Total_Base"].astype(float).values
])

r2_total_combined = np.concatenate([
    df2["Total_Our"].astype(float).values,
    df2["Total_Base"].astype(float).values
])

total_combined_ids = [f"{id_}_OurTotal" for id_ in common_ids] + [f"{id_}_BaseTotal" for id_ in common_ids]

summary_total_combined, icc_total_combined = compute_irr(
    r1_scores=r1_total_combined,
    r2_scores=r2_total_combined,
    unit_ids=total_combined_ids,
    measure_name="HR1_vs_HR2_Total_Pooled_Combined"
)

# =========================================================
# PART 2: 5-matrix combined — Our
# =========================================================
r1_our = []
r2_our = []
our_ids = []

for d in dims:
    for id_ in common_ids:
        r1_our.append(df1.loc[id_, f"{d}_Our"])
        r2_our.append(df2.loc[id_, f"{d}_Our"])
        our_ids.append(f"{id_}_{d}")

r1_our = np.array(r1_our, dtype=float)
r2_our = np.array(r2_our, dtype=float)

summary_our_5dim, icc_our_5dim = compute_irr(
    r1_scores=r1_our,
    r2_scores=r2_our,
    unit_ids=our_ids,
    measure_name="HR1_vs_HR2_Our_5Metrics_Combined"
)

# =========================================================
# PART 3: 5-matrix combined — Base
# =========================================================
r1_base = []
r2_base = []
base_ids = []

for d in dims:
    for id_ in common_ids:
        r1_base.append(df1.loc[id_, f"{d}_Base"])
        r2_base.append(df2.loc[id_, f"{d}_Base"])
        base_ids.append(f"{id_}_{d}")

r1_base = np.array(r1_base, dtype=float)
r2_base = np.array(r2_base, dtype=float)

summary_base_5dim, icc_base_5dim = compute_irr(
    r1_scores=r1_base,
    r2_scores=r2_base,
    unit_ids=base_ids,
    measure_name="HR1_vs_HR2_Base_5Metrics_Combined"
)

# =========================
# 6. Combine correlation summaries
# =========================
summary_all = pd.concat([
    summary_total_combined,
    summary_our_5dim,
    summary_base_5dim
], ignore_index=True)

summary_all = summary_all.round({
    "Pearson_r": 4,
    "Pearson_p": 6,
    "Spearman_r": 4,
    "Spearman_p": 6
})

print("\n=== Correlation Summary ===")
print(summary_all.to_string(index=False))

# =========================
# 7. Show ICC results
# =========================
print("\n=== ICC Results: Total pooled combined (Our + Base) ===")
print(icc_total_combined.round(4).to_string(index=False))

print("\n=== ICC Results: 5-matrix combined — Our ===")
print(icc_our_5dim.round(4).to_string(index=False))

print("\n=== ICC Results: 5-matrix combined — Base ===")
print(icc_base_5dim.round(4).to_string(index=False))

# =========================
# 8. Optional: save to Excel
# =========================
with pd.ExcelWriter("HR1_vs_HR2_interrater_results.xlsx", engine="openpyxl") as writer:
    summary_all.to_excel(writer, sheet_name="Correlation_Summary", index=False)
    icc_total_combined.to_excel(writer, sheet_name="ICC_Total_Pooled", index=False)
    icc_our_5dim.to_excel(writer, sheet_name="ICC_Our_5Dim", index=False)
    icc_base_5dim.to_excel(writer, sheet_name="ICC_Base_5Dim", index=False)

print("\nSaved: HR1_vs_HR2_interrater_results.xlsx")

Matched IDs: 10
Common IDs: ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']

Columns of icc_all for HR1_vs_HR2_Total_Pooled_Combined: ['Type', 'ICC', 'F', 'df1', 'df2', 'pval', 'CI95']


Columns of icc_all for HR1_vs_HR2_Our_5Metrics_Combined: ['Type', 'ICC', 'F', 'df1', 'df2', 'pval', 'CI95']


Columns of icc_all for HR1_vs_HR2_Base_5Metrics_Combined: ['Type', 'ICC', 'F', 'df1', 'df2', 'pval', 'CI95']


=== Correlation Summary ===
                          Measure  N  Pearson_r  Pearson_p  Spearman_r  Spearman_p
 HR1_vs_HR2_Total_Pooled_Combined 20     0.9242        0.0      0.8457    0.000003
 HR1_vs_HR2_Our_5Metrics_Combined 50     0.9666        0.0      0.8639    0.000000
HR1_vs_HR2_Base_5Metrics_Combined 50     0.8865        0.0      0.7904    0.000000

=== ICC Results: Total pooled combined (Our + Base) ===
    Type    ICC       F  df1  df2  pval         CI95
ICC(1,1) 0.9201 24.0187   19   20   0.0 [0.81, 0.97]
ICC(A,1) 0.9200 23.5045   19   19   0.0 [0.81, 

In [1]:
# =========================
# 0. Install package
# =========================
!pip install pingouin -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 5.2 MB/s eta 0:00:00


Avg HR vs Avg AI

In [3]:


# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import pingouin as pg
from IPython.display import display

# =========================
# 2. Load files
# =========================
ai_file1 = "Claude _Rater5Dim.csv"
ai_file2 = "GPT_Rater5Dim.csv"

hr_file1 = "HR1.csv"
hr_file2 = "HR2.csv"

ai1 = pd.read_csv(ai_file1)
ai2 = pd.read_csv(ai_file2)
hr1 = pd.read_csv(hr_file1)
hr2 = pd.read_csv(hr_file2)

# =========================
# 3. Clean columns and rows
# =========================
for df in [ai1, ai2, hr1, hr2]:
    df.columns = df.columns.str.strip()

ai1 = ai1[ai1["ID"].notna()].copy()
ai2 = ai2[ai2["ID"].notna()].copy()
hr1 = hr1[hr1["ID"].notna()].copy()
hr2 = hr2[hr2["ID"].notna()].copy()

for df in [ai1, ai2, hr1, hr2]:
    df["ID"] = df["ID"].astype(str).str.strip()

# =========================
# 4. Keep only needed columns
# =========================
dims = ["HRO", "MA", "PPA", "SPQ", "CER"]

score_cols = (
    [f"{d}_Our" for d in dims] +
    ["Total_Our"] +
    [f"{d}_Base" for d in dims] +
    ["Total_Base"]
)

keep_cols = ["ID"] + score_cols

ai1 = ai1[keep_cols].copy()
ai2 = ai2[keep_cols].copy()
hr1 = hr1[keep_cols].copy()
hr2 = hr2[keep_cols].copy()

# =========================
# 5. Match IDs within AI and HR separately
# =========================
ai1 = ai1.set_index("ID")
ai2 = ai2.set_index("ID")
hr1 = hr1.set_index("ID")
hr2 = hr2.set_index("ID")

common_ai_ids = sorted(ai1.index.intersection(ai2.index))
common_hr_ids = sorted(hr1.index.intersection(hr2.index))

ai1 = ai1.loc[common_ai_ids].copy()
ai2 = ai2.loc[common_ai_ids].copy()

hr1 = hr1.loc[common_hr_ids].copy()
hr2 = hr2.loc[common_hr_ids].copy()

print("Matched IDs inside AI raters:", len(common_ai_ids), common_ai_ids)
print("Matched IDs inside HR raters:", len(common_hr_ids), common_hr_ids)

# =========================
# 6. Create AI average per ID
# =========================
ai_avg = (ai1[score_cols] + ai2[score_cols]) / 2
ai_avg = ai_avg.reset_index()

print("\n=== AI Average table: top 5 rows ===")
display(ai_avg.head(5).round(3))

# =========================
# 7. Create HR average per ID
# =========================
hr_avg = (hr1[score_cols] + hr2[score_cols]) / 2
hr_avg = hr_avg.reset_index()

print("\n=== HR Average table: top 5 rows ===")
display(hr_avg.head(5).round(3))

# =========================
# 8. Match IDs between AI average and HR average
# =========================
ai_avg = ai_avg.set_index("ID")
hr_avg = hr_avg.set_index("ID")

common_ids = sorted(ai_avg.index.intersection(hr_avg.index))

ai_avg = ai_avg.loc[common_ids].copy()
hr_avg = hr_avg.loc[common_ids].copy()

print("\nMatched IDs between AI average and HR average:", len(common_ids))
print("Common IDs:", common_ids)

# =========================
# 9. Helper function for IRR
# =========================
def compute_irr(rater_a_scores, rater_b_scores, unit_ids, measure_name, rater_a_name="HR_Avg", rater_b_name="AI_Avg"):
    # Pearson
    pearson_r, pearson_p = pearsonr(rater_a_scores, rater_b_scores)

    # Spearman
    spearman_r, spearman_p = spearmanr(rater_a_scores, rater_b_scores)

    # ICC dataframe
    icc_df = pd.DataFrame({
        "ID": unit_ids * 2,
        "Rater": [rater_a_name] * len(unit_ids) + [rater_b_name] * len(unit_ids),
        "Score": list(rater_a_scores) + list(rater_b_scores)
    })

    icc_all = pg.intraclass_corr(
        data=icc_df,
        targets="ID",
        raters="Rater",
        ratings="Score"
    )

    icc_all = icc_all[["Type", "ICC", "F", "df1", "df2", "pval", "CI95"]].copy()

    summary = pd.DataFrame([{
        "Measure": measure_name,
        "N": len(unit_ids),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p
    }])

    return summary, icc_all

# =========================
# 10. 5-dimension combined — Our
# =========================
hr_our = []
ai_our = []
our_ids = []

for d in dims:
    for id_ in common_ids:
        hr_our.append(hr_avg.loc[id_, f"{d}_Our"])
        ai_our.append(ai_avg.loc[id_, f"{d}_Our"])
        our_ids.append(f"{id_}_{d}")

hr_our = np.array(hr_our, dtype=float)
ai_our = np.array(ai_our, dtype=float)

summary_our, icc_our = compute_irr(
    rater_a_scores=hr_our,
    rater_b_scores=ai_our,
    unit_ids=our_ids,
    measure_name="HR_Avg_vs_AI_Avg_Our_5Metrics_Combined"
)

# =========================
# 11. 5-dimension combined — Base
# =========================
hr_base = []
ai_base = []
base_ids = []

for d in dims:
    for id_ in common_ids:
        hr_base.append(hr_avg.loc[id_, f"{d}_Base"])
        ai_base.append(ai_avg.loc[id_, f"{d}_Base"])
        base_ids.append(f"{id_}_{d}")

hr_base = np.array(hr_base, dtype=float)
ai_base = np.array(ai_base, dtype=float)

summary_base, icc_base = compute_irr(
    rater_a_scores=hr_base,
    rater_b_scores=ai_base,
    unit_ids=base_ids,
    measure_name="HR_Avg_vs_AI_Avg_Base_5Metrics_Combined"
)

# =========================
# 12. Optional: 5-dim combined pooled (Our + Base)
# =========================
hr_pooled = np.concatenate([hr_our, hr_base])
ai_pooled = np.concatenate([ai_our, ai_base])
pooled_ids = [f"{x}_Our" for x in our_ids] + [f"{x}_Base" for x in base_ids]

summary_pooled, icc_pooled = compute_irr(
    rater_a_scores=hr_pooled,
    rater_b_scores=ai_pooled,
    unit_ids=pooled_ids,
    measure_name="HR_Avg_vs_AI_Avg_5Metrics_Pooled_OurPlusBase"
)

# =========================
# 13. Combine correlation summaries
# =========================
summary_all = pd.concat([summary_our, summary_base, summary_pooled], ignore_index=True)

summary_all = summary_all.round({
    "Pearson_r": 4,
    "Pearson_p": 6,
    "Spearman_r": 4,
    "Spearman_p": 6
})

print("\n=== Correlation Summary ===")
print(summary_all.to_string(index=False))

print("\n=== ICC Results: 5-matrix combined — Our ===")
print(icc_our.round(4).to_string(index=False))

print("\n=== ICC Results: 5-matrix combined — Base ===")
print(icc_base.round(4).to_string(index=False))

print("\n=== ICC Results: 5-matrix pooled (Our + Base) ===")
print(icc_pooled.round(4).to_string(index=False))

# =========================
# 14. Save outputs
# =========================
with pd.ExcelWriter("HRAvg_vs_AIAvg_5Dim_IRR.xlsx", engine="openpyxl") as writer:
    ai_avg.reset_index().round(3).to_excel(writer, sheet_name="AI_Average_Per_ID", index=False)
    hr_avg.reset_index().round(3).to_excel(writer, sheet_name="HR_Average_Per_ID", index=False)
    summary_all.to_excel(writer, sheet_name="Correlation_Summary", index=False)
    icc_our.to_excel(writer, sheet_name="ICC_Our_5Dim", index=False)
    icc_base.to_excel(writer, sheet_name="ICC_Base_5Dim", index=False)
    icc_pooled.to_excel(writer, sheet_name="ICC_Pooled_5Dim", index=False)

print("\nSaved: HRAvg_vs_AIAvg_5Dim_IRR.xlsx")

Matched IDs inside AI raters: 10 ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']
Matched IDs inside HR raters: 10 ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']

=== AI Average table: top 5 rows ===


,ID,HRO_Our,MA_Our,PPA_Our,SPQ_Our,CER_Our,Total_Our,HRO_Base,MA_Base,PPA_Base,SPQ_Base,CER_Base,Total_Base
0,R12,22.5,9.5,13.5,8.0,10.0,63.5,18.0,7.0,10.0,8.0,6.0,49.0
1,R15,23.0,9.5,13.0,9.0,9.0,63.5,17.5,7.5,10.5,7.0,6.5,49.0
2,R18,22.5,9.5,14.0,9.5,9.5,65.0,16.0,6.0,9.5,6.5,4.0,42.0
3,R26,22.0,10.0,14.5,10.0,10.0,66.5,16.5,8.0,11.0,7.5,6.5,49.5
4,R28,22.0,9.5,14.5,10.0,9.5,65.5,16.5,8.0,11.0,7.0,5.0,47.5



=== HR Average table: top 5 rows ===


,ID,HRO_Our,MA_Our,PPA_Our,SPQ_Our,CER_Our,Total_Our,HRO_Base,MA_Base,PPA_Base,SPQ_Base,CER_Base,Total_Base
0,R12,19.15,7.5,12.5,8.0,8.335,55.485,13.85,5.0,10.5,5.0,5.335,39.685
1,R15,20.80,8.5,15.0,9.0,8.000,61.300,14.00,5.5,13.0,6.0,5.005,43.505
2,R18,20.00,8.5,14.5,8.5,8.000,59.500,12.80,5.5,10.5,5.5,5.005,39.305
3,R26,20.90,8.0,15.0,9.5,8.000,61.400,14.65,5.5,11.5,7.0,4.005,42.655
4,R28,21.65,8.0,13.5,8.5,8.335,59.985,13.85,5.5,11.5,5.0,4.340,40.190



Matched IDs between AI average and HR average: 10
Common IDs: ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']

=== Correlation Summary ===
                                     Measure   N  Pearson_r  Pearson_p  Spearman_r  Spearman_p
      HR_Avg_vs_AI_Avg_Our_5Metrics_Combined  50     0.9771        0.0      0.7882         0.0
     HR_Avg_vs_AI_Avg_Base_5Metrics_Combined  50     0.9051        0.0      0.8478         0.0
HR_Avg_vs_AI_Avg_5Metrics_Pooled_OurPlusBase 100     0.9551        0.0      0.9140         0.0

=== ICC Results: 5-matrix combined — Our ===
    Type    ICC       F  df1  df2  pval         CI95
ICC(1,1) 0.9524 41.0380   49   50   0.0 [0.92, 0.97]
ICC(A,1) 0.9530 84.6939   49   49   0.0 [0.65, 0.98]
ICC(C,1) 0.9767 84.6939   49   49   0.0 [0.96, 0.99]
ICC(1,k) 0.9756 41.0380   49   50   0.0 [0.96, 0.99]
ICC(A,k) 0.9759 84.6939   49   49   0.0 [0.79, 0.99]
ICC(C,k) 0.9882 84.6939   49   49   0.0 [0.98, 0.99]

=== ICC Results: 5-matrix combined — Bas

In [6]:

# =========================
# Imports
# =========================
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import pingouin as pg

# =========================
# Load files
# =========================
hr1 = pd.read_csv("HR1.csv")
hr2 = pd.read_csv("HR2.csv")

ai1 = pd.read_csv("Claude _Rater5Dim.csv")
ai2 = pd.read_csv("GPT_Rater5Dim.csv")

# =========================
# Clean
# =========================
for df in [hr1, hr2, ai1, ai2]:
    df.columns = df.columns.str.strip()
    df.dropna(subset=["ID"], inplace=True)
    df["ID"] = df["ID"].astype(str).str.strip()

# =========================
# Keep only needed columns
# =========================
cols = ["ID", "Total_Our", "Total_Base"]

hr1 = hr1[cols].set_index("ID")
hr2 = hr2[cols].set_index("ID")

ai1 = ai1[cols].set_index("ID")
ai2 = ai2[cols].set_index("ID")

# =========================
# Match IDs across all
# =========================
common_ids = sorted(
    set(hr1.index) &
    set(hr2.index) &
    set(ai1.index) &
    set(ai2.index)
)

hr1 = hr1.loc[common_ids]
hr2 = hr2.loc[common_ids]
ai1 = ai1.loc[common_ids]
ai2 = ai2.loc[common_ids]

print("Matched IDs:", len(common_ids), common_ids)

# =========================
# Compute averages
# =========================
hr_avg = (hr1 + hr2) / 2
ai_avg = (ai1 + ai2) / 2

# =========================
# Build pooled TOTAL vectors (Our + Base)
# =========================
hr_total_pooled = np.concatenate([
    hr_avg["Total_Our"].values,
    hr_avg["Total_Base"].values
])

ai_total_pooled = np.concatenate([
    ai_avg["Total_Our"].values,
    ai_avg["Total_Base"].values
])

pooled_ids = (
    [f"{id_}_Our" for id_ in common_ids] +
    [f"{id_}_Base" for id_ in common_ids]
)

print("\nTotal pooled observations:", len(pooled_ids))

# =========================
# Pearson & Spearman
# =========================
pearson_r, pearson_p = pearsonr(hr_total_pooled, ai_total_pooled)
spearman_r, spearman_p = spearmanr(hr_total_pooled, ai_total_pooled)

print("\n=== HR Avg vs AI Avg (Total Pooled) ===")
print(f"Pearson r   : {pearson_r:.4f} (p={pearson_p:.4g})")
print(f"Spearman ρ  : {spearman_r:.4f} (p={spearman_p:.4g})")

# =========================
# ICC
# =========================
icc_df = pd.DataFrame({
    "ID": pooled_ids * 2,
    "Rater": ["HR_Avg"] * len(pooled_ids) + ["AI_Avg"] * len(pooled_ids),
    "Score": list(hr_total_pooled) + list(ai_total_pooled)
})

icc = pg.intraclass_corr(
    data=icc_df,
    targets="ID",
    raters="Rater",
    ratings="Score"
)

icc = icc[["Type", "ICC", "F", "df1", "df2", "pval", "CI95"]].round(4)

print("\n=== ICC Results (Total Pooled) ===")
print(icc.to_string(index=False))

Matched IDs: 10 ['R12', 'R15', 'R18', 'R26', 'R28', 'R32', 'R34', 'R36', 'R39', 'R5']

Total pooled observations: 20

=== HR Avg vs AI Avg (Total Pooled) ===
Pearson r   : 0.9083 (p=3.108e-08)
Spearman ρ  : 0.7267 (p=0.0002841)

=== ICC Results (Total Pooled) ===
    Type    ICC       F  df1  df2  pval         CI95
ICC(1,1) 0.7843  8.2729   19   20   0.0 [0.54, 0.91]
ICC(A,1) 0.7973 20.4186   19   19   0.0 [0.06, 0.94]
ICC(C,1) 0.9066 20.4186   19   19   0.0 [0.78, 0.96]
ICC(1,k) 0.8791  8.2729   19   20   0.0  [0.7, 0.95]
ICC(A,k) 0.8872 20.4186   19   19   0.0 [0.11, 0.97]
ICC(C,k) 0.9510 20.4186   19   19   0.0 [0.88, 0.98]
